In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab5.ipynb")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from resources.hashutils import *
rng_seed = 454

---

<h1><center>SDSE Lab 5 <br><br> Scikit-learn, logistic regression, feature selection, and regularization</center></h1>

---

In this lab we will build a model for diagnosing breast cancer from various measurements of a tumor. To do this we will use [scikit-learn](https://scikit-learn.org/stable/), which is a package for performing a host of machine learning tasks. We will learn about scikit-learn's train-test data splitter, its standard scaler, pipelines, cross-validation, and Lasso regularization. 

The lab has 11 parts.

**Prelminaries**

1. Load the data
2. Extract test data
3. Normalize the training data

**Simple logistic regression**

4. Most correlated feature
5. Train simple logistic regression
6. Create a scikit-learn pipeline
7. Evaluate the models with cross-validation

**Regularization**

8. Lasso regularized logistic regression
9. Choose the best model
10. Significant features
11. Evaluate the final model with test data

---

<h1><center><font color='purple'>Preliminaries</font><br></center></h1>

# 1. Choose the performance metric

This is a binary classification problem, so we must choose a binary performance metric for our evaluation criterion. Let's suppose that our stakeholder is the healthcare system as a whole -- not the patient and doctor, not the insurance companies. We obtain the information listed below about the costs. Here "positive" means malignant and "negative" means benign tumor. 
+ Incorrectly diagnosing a benign tumor as malignant  produces a loss of $2,000 relative to a correct diagnosis, due to unnecessary follow-up tests and administrative costs. 
+ A correct diagnosis of a positive case will result in early curative treatment costing $60,000.
+ Failing to detect a malignant tumor will result in more expenssive later tretments costing $200,000.

In [ ]:
def luf(y_true,y_pred):
    gamma = ...
    FP = ...
    FN = ...
    luf = ...
    return luf

In [ ]:
grader.check("q1")


# 2. Load the data

The [dataset](https://archive.ics.uci.edu/ml/datasets/Breast+Cancer+Wisconsin+(Diagnostic)) originates from the University of Wisconsin and is included in the [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/index.php), as well as in scikit-learn's collection of [toy datasets](https://scikit-learn.org/stable/datasets/toy_dataset.html). It can be loaded with scikit-learn's [load_breast_cancer](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html) method. 

```python
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer(as_frame=True).frame
```

Passing `as_frame=True` prompts the loader to return a pandas DataFrame. The raw dataset encodes a benign tumor as a 1 and a malignant tumor as a 0. We flip these tags so that the encoding agrees with the convention of a malignant tumor producing a "positive" outcome (1) and a benign tumor producing a "negative" outcome (0).

```python
data['target'] = 1-data['target']
```

Copy these lines of code into the cell below.

In [ ]:
...

In [ ]:
grader.check("q2")

Use `data.info()` to display a summary of the dataset. 

In [ ]:
...

# 3. Extract test data

The first step is to set aside a portion of the data for final testing. Use scikit-learn's [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) to create the testing and training datasets. 

Note that `train_test_split` takes the following arguments:
1. **First positional argument**: the input samples. These are all but the last column of the dataset. Hint: Use `data.iloc` to select all rows and all but the last column. 
2. **Second positional argument**: the output samples. This is the `'target'` column of the dataset.
3. **keyword argument `test_size`**: Set the portion of the dataset reserved for testing to 20% (0.2).
4. **keyword argument `random_state`**: Pass `random_state==rng_seed` in order to ensure reproducibility of the results. 

In [ ]:
from sklearn.model_selection import train_test_split
Xtrain, Xtest, ytrain, ytest = train_test_split(...,
                                                ...,
                                                ...,
                                                ... )

In [ ]:
grader.check("q3")

# 4. Normalize the inputs

Many models, including linear and logistic regression, benefit from a "normalization" of their input values. In the case of linear regression, normalization helps because it improves the conditioning number of the extended design matrix $\mathbb{X}$.  

To *normalize* a column $X_i$ of a dataset is to subtract out its mean and divide by its standard deviation. This produces a new column $\widetilde{X}_i$.

\begin{equation*}
\widetilde{X}_i = \frac{X_i-\text{mean}(X_i)}{\text{std}(X_i)}
\end{equation*}

Models whose training procedure is *not* affected by normalization are called "scale invariant". Logistic regression is *not* scale invariant, meaning that normalization *is* beneficial. 

We will use scikit-learn's [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) to perform the normalization of the training input data (`Xtrain`). 

Follow these steps:

1. Create a standard scaler object with
```python
scaler = StandardScaler()
```

2. Call the `fit_transform` method of the `scaler` and pass in the DataFrame we wish to normalize. Keep the result as `Xtilde`. Notice that `Xtilde` is a 2D NumPy array. 
3. Put the normalized data into a DataFrame. You can do this with the constructor `pd.DataFrame` and passing in `Xtilde`. Also set the index to `Xtrain`'s index using the `index` keyword argument, and set the column names if DataFrame to `Xtrain.column` using the `columns` keyword argument. Store the resulting DataFrame as `Xtrain_norm`. 

In [ ]:
from sklearn.preprocessing import StandardScaler

# Create a scaler object
scaler = StandardScaler()

# Use the fit_transform method to perform the normalization of columns
Xtilde = ...

# Put the transformed data into a DataFrame similar to Xtrain
Xtrain_norm = ...

In [ ]:
grader.check("q4")


---

<h1><center><font color='purple'> Simple logistic regression</font><br></center></h1>



# 5. Most correlated feature

Our first model will be a simple logistic regression model based on the single feature that best correlates with the output. Find this feature and save its column name  to `best_single_feature`. 

To do this, follow these steps:

1. Find the sample correlation of each input column with the target. This must be done on the training data alone (not including the test data) to avoid data leakage. The input DataFrame's `corrwith` method is useful for this purpose. Call `corrwith` of `Xtrain` and pass in `ytain`. Call the resulting pandas Series `corr`. Inspect it using `print`. 

2. Next we need to find the feature whose sample correlation with the output is of greatest magnitude. First take the absolute value of the correlations by calling the `abs()` method of `corr`. Then sort the correlations in descending order by calling `sort_values` on the result and passing in keyword argument `ascending=False`. Finally select the index of the first (ie. the largest) entry with `.index[0]`. These three steps may be concatendated.

In [ ]:
# 1. Correlation of each of the input columns with the target.
corr = ...

# 2. Take absolute value, sort in descending order, keep the first index.
...
best_single_feature = ...

In [ ]:
grader.check("q5")

# 6. Train simple logistic regression

Next we train a simple logistic regression model for the feature that was selected in the previous part. For this we will use scikit-learn's  [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) class.


We will do this on both the un-normalized and the normalize data, in order to check whether the result improves, as expected. 

Follow these steps for the un-normalized data.

1. Create a logistic regression model using the `LogisticRegression` constructor. Pass in `random_state=rng_seed` to ensure repeatability of the results. Name the object `model_unnorm`.
2. Train the logistic regression model using `model_unnorm`'s `fit` function. This function takes two positional arguments:
    + **First positional argument**: the input data. Use the column of `Xtrain` corresponding to the most correlated feature.
    + **Second positional argument**: the output data  `ytrain`.
3. Use 4-fold cross-validation to evaluate the model. Scikit-learn offers a `cross_val_score` method for this purpose. This takes the following input arguments.
    + **First positional argument**: the model, in this case `model_unnorm`.
    + **Second positional argument**: the input data.
    + **Third positional argument**: the output data.
    + **keyword argument** `cv`: the number of folds, in this case 4.
    + **keyword argument** `scoring`: the performance metric. We will use the `luf` defined at the beginning of this lab as the performance metric. For this to work, we must create a Scikit-learn scoring object by passing our `luf` function to Scikit-learn's `make_scorer` function. In other words, you should pass `scoring=make_scorer(luf)`. 

    `cross_val_score` returns the performance for each fold as a NumPy array. Keep the *average* of these scores in the variable name `model_unnorm_perf`.

Repeat these steps using the *normalized* input data `Xtrain_norm`. Save  model as `model_norm`.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_val_score


# 1. Create the logistic regression model with fixed random seed
model_unnorm = ...

# 2. Train the model using the most correlated feature only
...

# 3. Evaluate the model using 4-fold cross validation.
...
model_unnorm_perf = ...

# 4. Repeat with the normalized data
model_norm = ...
...
model_norm_perf = ...

In [ ]:
grader.check("q6")

# 7. Create a scikit-learn pipeline

Scikit-learn provides a *pipeline* class that collects all of the preprocessing, feature transformation, and modeling components into a single object with `fit` and `predict` methods. You can  read the documentation on [pipelines](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) to learn more. 

Each component in the pipeline is identified with a string name. The following code creates a pipeline with a `StandardScaler` labeled as `scaler`, followed by a logistic regression model labeled as `logreg`.

``` python
pipeline = Pipeline([('scaler', StandardScaler()), 
                     ('logreg', LogisticRegression(random_state=rng_seed)) ])
```

Create this pipeline and train it on the `best_single_feature` of the un-normalized dataset (`Xtrain`,`ytrain`) using the `fit` method. 

Evaluate the pipleine with 4-fold cross validation, using the linear utility function as the metric. You can do this identically as in the previous part, but passing the pipeline instead of the model as the first positional argument of `cross_val_score`. Savd the average of the four scores as `pipeline_perf`.

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline(...)
pipeline.fit(...) 
pipeline_perf = ...

In [ ]:
grader.check("q7")


---

<h1><center><font color='purple'>Regularization</font><br></center></h1>




# 8. Lasso regularized logistic regression

Regularization is a method for avoiding overfitting by penalizing the complexity of the model in the training process. Lasso regularization in particular penalizes the sum of the absolute values of the slope parameters. It has the property that it tends to "zero out" coefficients as the penalty $\lambda$ increases. This gives it an additional role as a feature selection technique. 

In this part we will train a Lasso regularized logistic regression model. Instead of $\lambda$, Scikit-learn uses the `C` parameter of the `LogisticRegression` constructor, which is the proportional to the inverse of $\lambda$. Hence, increasing `C` results in *less* regularization, and generally larger parameter values. 

Write code that loops through a **logarithmically spaced** array of 20 regularization parameters `C` ranging from $10^{-2}$ to $10^{2}$. Create this array using [np.logspace](https://numpy.org/devdocs/reference/generated/numpy.logspace.html) and store it as `Cs`.

For each value in the array, the code should 
1. Train a logistic regression pipeline. 
2. Evaluate the pipeline using 4-fold cross-validation and our chosen linear utility metric.
3. Store the model in a list of models called `models`. 
4. Store the performance values in a NumPy array called `perfs`.

Your pipeline should have two componenents: a `StandardScaler` for normalizing the data, followed by a `LogisticRegression` model. When building the `LogisticRegression` model, pass these keyword arguments to the constructor: 
+ `C`: the value of the regularization parameter
+ `l1_ratio=1`: for Lasso (as opposed to Ridge) regularization.
+ `solver='liblinear'`: this is the best numerical solver to use for training logistic regression with lasso regularization.
+ `random_state=rng_seed`: set the random seed to ensure reproducibility.

In [ ]:
# Create a logarithmically spaced array of 20 values ranging from 0.01 to 100
Cs = ,,,

# Initialize perfs to an empty array of 20 values
perfs = ...

# initialize models to an empty list
models = ...


for c, C in enumerate(Cs):   

    # Create a pipeline model 
    model = ...
    
    # Store the model in the models list
    ...

    # Fit the model using the training data
    ...  

    # Estimate performance using 4-fold cross-validation.
    ...

    # Save the average of the four folds in the perfs array
    perfs[c] = ...

In [ ]:
grader.check("q8")

# 9. Choose the best model

Next we select the model with the model with the best score. Follow the steps in the code. 

In [ ]:
# 1. Set `cstar` to the index of the best performing regularization value
cstar = ...

# 2. Set `perf_star` to the corresponding performance score
perf_star = ...

# The next bit of code extracts the coefficients of the logistic regression for each of the 20 values of `C`. 
# This is stored in `theta`, which is a (20,30) array. (30 is the number of features)
theta = np.vstack([model.named_steps['logreg'].coef_[0,:] for model in models])

# 3. Plot the score as a function of `C`. (done already)
fig, ax = plt.subplots(figsize=(8,8),nrows=2,sharex=True)
ax[0].semilogx(Cs,perfs,'o-',color='b',linewidth=2)
ax[0].semilogx(Cs[cstar],perf_star,'*',color='m',markersize=14)
ax[0].grid(linestyle=':')
ax[0].set_ylabel('performance',fontsize=12)

# 4. In a single plot, plot the 30 coefficients as a fucntion of `C`.
ax[1].semilogx(Cs,theta)
ax[1].grid(linestyle=':')
ax[1].set_xlabel('C',fontsize=16)

In [ ]:
grader.check("q9")

# 10. Significant features

The plot below shows the coefficients for the best-case regularized logistic regression found in the previous part. Notice that many of these coefficients have been set to zero. 

In [ ]:
theta_star = theta[cstar,:]

plt.figure(figsize=(10,3))
plt.stem(np.abs(theta_star));

Follow the instructions in the cell that follows. 

In [ ]:
features = Xtrain.columns

# 1. Set `best_features` to the set of feature names corresponding to non-zero coefficients in the plot above. 
best_features = ...

# 2. Set `max_theta_feature` to the feature name corresponding to the coefficient with maximum absolute value. 
max_theta_feature = ...

# 3. Save the selected lasso model to the variable `lasso_model`.
lasso_model = ...

In [ ]:
grader.check("q10")

# 11. Evaluate the final model with test data

Use the test dataset to evaluate the performance of the selected lasso model. You can use the `luf` method created in part 1 of this lab for this. 

In [ ]:
# Use the model's `predict` method to predict outputs for the test inputs
yhat = ...

# Use the `luf` function to evaluate those predictions against the true test output. 
lasso_test = ...

In [ ]:
grader.check("q11")

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

Please submit the .zip file to Gradescope.

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)